In [33]:
"""
Preprocess and scale captured lab data (target domain). Reuse scaler from CICIDS2017,
and calculate covariance statistics.
"""

### Imports ###
import json
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.covariance import LedoitWolf

# Load shared feature-space artifacts in a single, validated format.
def load_feature_order(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict):
        if "features" not in payload:
            raise ValueError(
                f"Expected key 'features' in {path} when JSON object is provided."
            )
        feature_order = list(payload["features"])
    elif isinstance(payload, list):
        feature_order = list(payload)
    else:
        raise ValueError(
            f"Unsupported shared feature space format in {path}: "
            f"{type(payload).__name__}"
        )

    if not feature_order:
        raise ValueError(f"Shared feature space in {path} is empty")

    return feature_order

In [34]:
### Import and parse JSON files ###

# Creates a Path object pointing to the target-domain JSON directory.
data_dir = Path("data/raw/target")

# Read one target JSON file and return all flow records as a DataFrame.
# Expected format: {"flows": [{...}, {...}, ...]}
def read_target_json_flows(file_path: Path):
    with open(file_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if not isinstance(payload, dict):
        raise ValueError(
            f"Expected JSON object in {file_path}, got {type(payload).__name__}"
        )

    if "flows" not in payload:
        raise ValueError(f"Missing required key 'flows' in {file_path}")

    flows = payload["flows"]
    if not isinstance(flows, list):
        raise ValueError(
            f"Expected 'flows' to be a list in {file_path}, got {type(flows).__name__}"
        )

    if not all(isinstance(flow, dict) for flow in flows):
        raise ValueError(
            f"All entries in 'flows' must be JSON objects in {file_path}"
        )

    flow_df = pd.DataFrame.from_records(flows)
    expected_count = len(flows)

    if len(flow_df) != expected_count:
        raise ValueError(
            f"Flow count mismatch while loading {file_path}: "
            f"expected {expected_count}, loaded {len(flow_df)}"
        )

    return flow_df, expected_count

# Target domain spans multiple JSON exports; load and concatenate all of them.
json_files = sorted(data_dir.glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {data_dir}")

per_file_counts = {}
dfs = []
total_expected_flows = 0

for file_path in json_files:
    flow_df, expected_count = read_target_json_flows(file_path)
    per_file_counts[file_path.name] = expected_count
    total_expected_flows += expected_count
    dfs.append(flow_df)

df = pd.concat(dfs, ignore_index=True, sort=False) if dfs else pd.DataFrame()

# Integrity check: ensure no flows were lost across concatenation.
if len(df) != total_expected_flows:
    raise ValueError(
        f"Total flow mismatch after concatenation: expected {total_expected_flows}, got {len(df)}"
    )

# Display a quick shape check and preview rows.
print(f"Loaded and concatenated {len(json_files)} target JSON files:")
for file_name in [p.name for p in json_files]:
    print(f"  - {file_name}: {per_file_counts[file_name]} flows")
print("Dataset shape:", df.shape)
print("Total flows loaded:", len(df))
print("Flow integrity check passed: no flows lost during ingestion.")
df.head()

Loaded and concatenated 8 target JSON files:
  - capture_20260414_185327.json: 12556 flows
  - capture_20260415_131451.json: 3311 flows
  - capture_20260415_133820.json: 45695 flows
  - capture_20260417_131850.json: 8397 flows
  - capture_20260417_142847.json: 9129 flows
  - capture_20260417_152940.json: 4678 flows
  - capture_20260419_094747.json: 49955 flows
  - capture_20260419_130742.json: 2172 flows
Dataset shape: (135893, 70)
Total flows loaded: 135893
Flow integrity check passed: no flows lost during ingestion.


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,-0.354687,-0.470908,-0.010425,-0.01095,-0.044950,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
1,-0.354687,-0.470914,-0.011684,-0.01095,-0.051373,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
2,-0.354687,-0.470868,-0.002872,-0.01095,-0.006417,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
3,-0.354687,-0.470868,-0.002872,-0.01095,-0.006417,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
4,-0.354687,-0.470908,-0.010425,-0.01095,-0.044950,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764


In [35]:
### [Diagnostic] ###

# DataFrame shape snapshot at this stage of preprocessing.
print(f"DataFrame shape: {df.shape} (rows={len(df):,}, cols={len(df.columns):,})")

# Feature space: list every column present in the compiled target dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

DataFrame shape: (135893, 70) (rows=135,893, cols=70)
Feature/column count: 70
Feature space (all dataset columns):
- Destination Port
- Flow Duration
- Total Fwd Packets
- Total Backward Packets
- Total Length of Fwd Packets
- Total Length of Bwd Packets
- Fwd Packet Length Max
- Fwd Packet Length Min
- Fwd Packet Length Mean
- Fwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packet Length Min
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Flow Bytes/s
- Flow Packets/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Total
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Total
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Fwd URG Flags
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Bwd Packets/s
- Min Packet Length
- Max Packet Length
- Packet Length Mean
- Packet Length Std
- Packet Length Variance
- FIN Flag Count
- SYN Flag Count
- RST Flag Count
- PSH Flag Count
- ACK Flag Count
- URG Flag Count

In [36]:
### Data sanitization ###

rows_before = len(df)

# Remove irrelevant columns.
df = df.drop(columns=["Flow ID", "Src IP", "Src Port", "Dst IP", "Timestamp"], errors="ignore")

# Remove duplicate flows.
# rows_before_dedup = len(df)
# df.drop_duplicates(inplace=True)
# rows_removed_dedup = rows_before_dedup - len(df)
# print(f"  (Deduplication: {rows_removed_dedup:,} rows removed, {len(df):,} rows remaining)")

# Remove leading/trailing spaces from all column names.
df.rename(columns=lambda x: x.strip(), inplace=True)

rows_after = len(df)
total_rows_removed = rows_before - rows_after
print(f"\nTotal rows removed during sanitization: {total_rows_removed:,} ({total_rows_removed/rows_before*100:.2f}%)")
print(f"Final row count: {rows_after:,}")


Total rows removed during sanitization: 0 (0.00%)
Final row count: 135,893


In [37]:
### [Diagnostic] ###

# DataFrame shape snapshot at this stage of preprocessing.
print(f"DataFrame shape: {df.shape} (rows={len(df):,}, cols={len(df.columns):,})")

# Feature space: list every column present in the compiled target dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

DataFrame shape: (135893, 70) (rows=135,893, cols=70)
Feature/column count: 70
Feature space (all dataset columns):
- Destination Port
- Flow Duration
- Total Fwd Packets
- Total Backward Packets
- Total Length of Fwd Packets
- Total Length of Bwd Packets
- Fwd Packet Length Max
- Fwd Packet Length Min
- Fwd Packet Length Mean
- Fwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packet Length Min
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Flow Bytes/s
- Flow Packets/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Total
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Total
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Fwd URG Flags
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Bwd Packets/s
- Min Packet Length
- Max Packet Length
- Packet Length Mean
- Packet Length Std
- Packet Length Variance
- FIN Flag Count
- SYN Flag Count
- RST Flag Count
- PSH Flag Count
- ACK Flag Count
- URG Flag Count

In [38]:
### Inject attack samples into data (1) ###

ATTACK_CSV_PATH = Path("data/raw/injected-attacks/attack_holdout.csv")
SCALER_PATH = Path("models/imported-scaler.joblib")

if not ATTACK_CSV_PATH.exists():
    raise FileNotFoundError(f"Attack CSV not found: {ATTACK_CSV_PATH}")
if not SCALER_PATH.exists():
    raise FileNotFoundError(f"Imported scaler not found: {SCALER_PATH}")

# Load attack data from holdout CSV.
attack_df = pd.read_csv(ATTACK_CSV_PATH)
attack_df.rename(columns=lambda x: x.strip(), inplace=True)

label_col = "Label" if "Label" in attack_df.columns else None
feature_cols = [c for c in attack_df.columns if c != label_col]

# Diagnostics: schema, shape, and label frequencies.
print("Attack data schema (raw):")
for col_name, dtype in attack_df.dtypes.items():
    print(f"- {col_name}: {dtype}")

print(f"\nRaw attack data shape: {attack_df.shape}")
print(f"Feature-only shape (no label): {attack_df[feature_cols].shape}")
nan_rows = int(attack_df[feature_cols].isna().any(axis=1).sum())
print(f"Rows with at least one NaN in feature set: {nan_rows:,}")

if label_col:
    print("\nAttack class frequencies:")
    print(attack_df[label_col].value_counts(dropna=False).sort_values(ascending=False))

Attack data schema (raw):
- Destination Port: int64
- Flow Duration: int64
- Total Fwd Packets: int64
- Total Backward Packets: int64
- Total Length of Fwd Packets: float64
- Total Length of Bwd Packets: float64
- Fwd Packet Length Max: float64
- Fwd Packet Length Min: float64
- Fwd Packet Length Mean: float64
- Fwd Packet Length Std: float64
- Bwd Packet Length Max: float64
- Bwd Packet Length Min: float64
- Bwd Packet Length Mean: float64
- Bwd Packet Length Std: float64
- Flow Bytes/s: float64
- Flow Packets/s: float64
- Flow IAT Mean: float64
- Flow IAT Std: float64
- Flow IAT Max: float64
- Flow IAT Min: float64
- Fwd IAT Total: float64
- Fwd IAT Mean: float64
- Fwd IAT Std: float64
- Fwd IAT Max: float64
- Fwd IAT Min: float64
- Bwd IAT Total: float64
- Bwd IAT Mean: float64
- Bwd IAT Std: float64
- Bwd IAT Max: float64
- Bwd IAT Min: float64
- Fwd PSH Flags: int64
- Bwd PSH Flags: int64
- Fwd URG Flags: int64
- Bwd URG Flags: int64
- Fwd Header Length: int64
- Bwd Header Length:

In [39]:
### Inject attack samples into data (2) ###

# define attack caps
CLASS_CAPS = {
    "DDoS": 15000,
    "Infiltration": 7500,
    "DoS Hulk": 7500,
    "Bot": 5000,
    "SSH-Patator": 5000,
    "DoS GoldenEye": 5000,
    "DoS slowloris": 1000,
    "Web Attack - Brute Force": 555,
    "Web Attack - XSS": 228,
    "Web Attack - Sql Injection": 84,
    "DoS Slowhttptest": 55,
    "FTP-Patator": 50,
}

# perform per class reductions
if label_col is None:
    raise ValueError("Could not find 'Label' column in attack dataframe.")

attack_counts_before = attack_df[label_col].value_counts(dropna=False)
uncapped_labels = [label for label in attack_counts_before.index if label not in CLASS_CAPS]
if uncapped_labels:
    raise ValueError(
        "Found attack classes without defined caps: "
        f"{uncapped_labels}"
    )

reduced_frames = []
for attack_class, class_cap in CLASS_CAPS.items():
    class_slice = attack_df[attack_df[label_col] == attack_class]
    reduced_frames.append(class_slice.iloc[: int(class_cap)])

attack_df = pd.concat(reduced_frames, ignore_index=True)
attack_counts_after = attack_df[label_col].value_counts(dropna=False)

print("Applied per-class attack caps:")
for attack_class, class_cap in CLASS_CAPS.items():
    before_count = int(attack_counts_before.get(attack_class, 0))
    after_count = int(attack_counts_after.get(attack_class, 0))
    print(f"- {attack_class}: {before_count:,} -> {after_count:,} (cap={class_cap:,})")
print(f"Reduced attack sample count: {len(attack_df):,}")

# perform feature space alignment (align attack samples' features to that of lab
# data)
FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {FEATURE_LIST_PATH}")

shared_features = load_feature_order(FEATURE_LIST_PATH)

if label_col is None:
    raise ValueError("Could not find 'Label' column in attack dataframe.")

attack_feature_df = attack_df.drop(columns=[label_col], errors="ignore").copy()

missing_features = [feature for feature in shared_features if feature not in attack_feature_df.columns]
extra_features = [column for column in attack_feature_df.columns if column not in shared_features]

for feature in missing_features:
    attack_feature_df[feature] = 0.0

if extra_features:
    attack_feature_df = attack_feature_df.drop(columns=extra_features)

# Enforce canonical feature order exactly as specified by shared_feature_space.json.
attack_feature_df = attack_feature_df.reindex(columns=shared_features)

attack_df = pd.concat([
    attack_feature_df,
    attack_df[[label_col]].reset_index(drop=True),
] , axis=1)

print(f"Loaded shared feature space from {FEATURE_LIST_PATH}")
print(f"Attack features aligned to canonical order: {len(shared_features)} columns")
print(f"Dropped extra attack columns: {len(extra_features)}")
print(f"Added missing attack columns with 0 fill: {len(missing_features)}")
if missing_features:
    print("Missing columns added:")
    for feature in missing_features:
        print(f"- {feature}")

Applied per-class attack caps:
- DDoS: 277,806 -> 15,000 (cap=15,000)
- Infiltration: 88,527 -> 7,500 (cap=7,500)
- DoS Hulk: 83,665 -> 7,500 (cap=7,500)
- Bot: 51,142 -> 5,000 (cap=5,000)
- SSH-Patator: 50,143 -> 5,000 (cap=5,000)
- DoS GoldenEye: 41,406 -> 5,000 (cap=5,000)
- DoS slowloris: 9,908 -> 1,000 (cap=1,000)
- Web Attack - Brute Force: 555 -> 555 (cap=555)
- Web Attack - XSS: 228 -> 228 (cap=228)
- Web Attack - Sql Injection: 84 -> 84 (cap=84)
- DoS Slowhttptest: 55 -> 55 (cap=55)
- FTP-Patator: 50 -> 50 (cap=50)
Reduced attack sample count: 46,972
Loaded shared feature space from data/processed/shared_feature_space.json
Attack features aligned to canonical order: 70 columns
Dropped extra attack columns: 8
Added missing attack columns with 0 fill: 1
Missing columns added:
- Fwd Header Length.1


In [40]:
### Inject attack samples into data (3) ###

# scale attack data using imported scaler in models/imported-scaler.joblib
IMPORTED_SCALER_PATH = Path("models/imported-scaler.joblib")
if not IMPORTED_SCALER_PATH.exists():
    raise FileNotFoundError(f"Imported scaler not found at {IMPORTED_SCALER_PATH}")

if label_col is None:
    raise ValueError("Could not find 'Label' column in attack dataframe.")

imported_scaler = joblib.load(IMPORTED_SCALER_PATH)
attack_feature_cols = [c for c in attack_df.columns if c != label_col]
attack_features_to_scale = attack_df[attack_feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)

if hasattr(imported_scaler, "feature_names_in_"):
    scaler_feature_order = list(imported_scaler.feature_names_in_)
    missing_for_scaler = [c for c in scaler_feature_order if c not in attack_features_to_scale.columns]
    if missing_for_scaler:
        raise ValueError(
            "Attack feature set is missing columns required by imported scaler: "
            f"{missing_for_scaler}"
        )
    attack_features_to_scale = attack_features_to_scale.reindex(columns=scaler_feature_order)

scaled_attack_array = imported_scaler.transform(attack_features_to_scale)
scaled_attack_df = pd.DataFrame(
    scaled_attack_array,
    columns=attack_features_to_scale.columns,
    index=attack_df.index,
 )
scaled_attack_df[label_col] = attack_df[label_col].values

# add Label column to benign-only data samples
BENIGN_LABEL_CANDIDATES = ["Label", " Label", "label"]
benign_label_col = next((c for c in BENIGN_LABEL_CANDIDATES if c in df.columns), None)
if benign_label_col is None:
    benign_label_col = "Label"
    df[benign_label_col] = "Benign"
else:
    df[benign_label_col] = "Benign"

# consolidate attack samples into main benign-only data
benign_df = df.copy()
if benign_label_col != label_col:
    benign_df = benign_df.rename(columns={benign_label_col: label_col})

benign_df = benign_df.assign(_source="benign")
scaled_attack_df = scaled_attack_df.assign(_source="attack")

combined_df = pd.concat([benign_df, scaled_attack_df], ignore_index=True, sort=False)
df = combined_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

# verify benign/attack proportion
source_counts = df["_source"].value_counts()
benign_count = int(source_counts.get("benign", 0))
attack_count = int(source_counts.get("attack", 0))
total_count = benign_count + attack_count

if total_count == 0:
    raise ValueError("Consolidated dataset is empty after injection.")

benign_pct = benign_count / total_count * 100
attack_pct = attack_count / total_count * 100

print(f"Loaded imported scaler from {IMPORTED_SCALER_PATH}")
print(f"Scaled attack samples: {len(scaled_attack_df):,}")
print(f"Consolidated dataset size: {len(df):,}")
print(f"Benign/Attack counts: {benign_count:,}/{attack_count:,}")
print(f"Benign/Attack ratio: {benign_pct:.2f}%/{attack_pct:.2f}%")

# remove helper column from final dataset
df = df.drop(columns=["_source"])

/Users/amazlumyan/Desktop/GitHub/domain-adaptation-poc/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Loaded imported scaler from models/imported-scaler.joblib
Scaled attack samples: 46,972
Consolidated dataset size: 182,865
Benign/Attack counts: 135,893/46,972
Benign/Attack ratio: 74.31%/25.69%


In [41]:
### Feature-space alignment verification ###
# (verify selected features exactly match predetermined shared feature space)

FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {FEATURE_LIST_PATH}")

shared_features = load_feature_order(FEATURE_LIST_PATH)

LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

current_features = [c for c in df.columns if c != label_col]
shared_feature_set = set(shared_features)
current_feature_set = set(current_features)

missing_features = sorted(shared_feature_set - current_feature_set)
extra_features = sorted(current_feature_set - shared_feature_set)
order_matches = current_features == shared_features

print(f"Loaded shared feature space from {FEATURE_LIST_PATH}")
print(f"Expected feature count: {len(shared_features)}")
print(f"Observed feature count: {len(current_features)}")
print(f"Missing features: {len(missing_features)}")
print(f"Extra features: {len(extra_features)}")
print(f"Exact order match: {order_matches}")

if missing_features:
    print("Missing feature names (first 10):")
    for feature in missing_features[:10]:
        print(f"- {feature}")
if extra_features:
    print("Extra feature names (first 10):")
    for feature in extra_features[:10]:
        print(f"- {feature}")

if not order_matches and not missing_features and not extra_features:
    mismatch_preview = [
        (idx, current_features[idx], shared_features[idx])
        for idx in range(min(len(current_features), len(shared_features)))
        if current_features[idx] != shared_features[idx]
    ][:10]
    print("Order mismatches (index, observed, expected):")
    for idx, observed, expected in mismatch_preview:
        print(f"- {idx}: {observed} != {expected}")

if missing_features or extra_features or not order_matches:
    raise ValueError(
        "Feature-space verification failed. Ensure the dataset columns exactly "
        "match shared_feature_space.json in both membership and order."
    )

print("Feature-space verification passed: dataset matches shared schema exactly.")

Loaded shared feature space from data/processed/shared_feature_space.json
Expected feature count: 70
Observed feature count: 70
Missing features: 0
Extra features: 0
Exact order match: True
Feature-space verification passed: dataset matches shared schema exactly.


In [42]:
### Label-space alignment ###
# (align labels according to predetermined shared label space)

SHARED_LABEL_SPACE_PATH = Path("data/processed/shared_label_space.json")
if not SHARED_LABEL_SPACE_PATH.exists():
    raise FileNotFoundError(
        f"Shared label space file not found at {SHARED_LABEL_SPACE_PATH}"
    )

with open(SHARED_LABEL_SPACE_PATH, "r", encoding="utf-8") as f:
    shared_label_payload = json.load(f)

if isinstance(shared_label_payload, dict):
    if "labels" not in shared_label_payload:
        raise ValueError(
            f"Expected key 'labels' in {SHARED_LABEL_SPACE_PATH} when JSON object is provided."
        )
    shared_label_space = [str(label).strip() for label in shared_label_payload["labels"]]
elif isinstance(shared_label_payload, list):
    shared_label_space = [str(label).strip() for label in shared_label_payload]
else:
    raise ValueError(
        f"Unsupported shared label space format in {SHARED_LABEL_SPACE_PATH}: "
        f"{type(shared_label_payload).__name__}"
    )

if not shared_label_space:
    raise ValueError(f"Shared label space in {SHARED_LABEL_SPACE_PATH} is empty")

LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

observed_labels_raw = df[label_col].astype("string").str.strip()
missing_label_mask = observed_labels_raw.isna() | observed_labels_raw.eq("")
missing_label_count = int(missing_label_mask.sum())
if missing_label_count > 0:
    raise ValueError(
        f"Found {missing_label_count} missing/blank label values in consolidated dataset."
    )

observed_label_space = sorted(observed_labels_raw[~missing_label_mask].unique().tolist())
expected_label_space = sorted(set(shared_label_space))

missing_classes = sorted(set(expected_label_space) - set(observed_label_space))
unexpected_classes = sorted(set(observed_label_space) - set(expected_label_space))

print(f"Loaded shared label space from {SHARED_LABEL_SPACE_PATH}")
print(f"Expected label classes: {len(expected_label_space)}")
print(f"Observed label classes: {len(observed_label_space)}")
print(f"Missing expected classes: {len(missing_classes)}")
print(f"Unexpected classes: {len(unexpected_classes)}")

if missing_classes:
    print("Missing expected classes:")
    for cls in missing_classes:
        print(f"- {cls}")
if unexpected_classes:
    print("Unexpected observed classes:")
    for cls in unexpected_classes:
        print(f"- {cls}")

if missing_classes or unexpected_classes:
    raise ValueError(
        "Consolidated dataset label space does not match shared_label_space.json."
    )

print("Label-space verification passed: consolidated labels match shared schema exactly.")

Loaded shared label space from data/processed/shared_label_space.json
Expected label classes: 13
Observed label classes: 13
Missing expected classes: 0
Unexpected classes: 0
Label-space verification passed: consolidated labels match shared schema exactly.


In [43]:
### Diagnostics ###

# Print feature space, label space, class distributions, benign/attack proportions,
# and total data shape.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

feature_columns = [c for c in df.columns if c != label_col]
print(f"Total data shape: {df.shape} (rows={len(df):,}, cols={len(df.columns):,})")
print(f"Feature count (excluding label): {len(feature_columns)}")
print("Feature space:")
for feature_name in feature_columns:
    print(f"- {feature_name}")

labels = df[label_col].astype("string").str.strip()
valid_labels = labels[~labels.isna() & ~labels.eq("")]
label_space = sorted(valid_labels.unique().tolist())
print(f"\nLabel-space size: {len(label_space)}")
print("Label space:")
for cls in label_space:
    print(f"- {cls}")

class_counts = valid_labels.value_counts().sort_values(ascending=False)
print("\nClass distribution:")
for cls, count in class_counts.items():
    pct = count / len(df) * 100
    print(f"- {cls}: {int(count):,} ({pct:.2f}%)")

benign_mask = valid_labels.str.lower().eq("benign")
benign_count = int(benign_mask.sum())
attack_count = int(len(valid_labels) - benign_count)
valid_total = benign_count + attack_count
if valid_total == 0:
    raise ValueError("No valid labels found for benign/attack diagnostics.")

benign_pct = benign_count / valid_total * 100
attack_pct = attack_count / valid_total * 100
print("\nBenign/Attack proportions:")
print(f"- Benign: {benign_count:,} ({benign_pct:.2f}%)")
print(f"- Attack: {attack_count:,} ({attack_pct:.2f}%)")

Total data shape: (182865, 71) (rows=182,865, cols=71)
Feature count (excluding label): 70
Feature space:
- Destination Port
- Flow Duration
- Total Fwd Packets
- Total Backward Packets
- Total Length of Fwd Packets
- Total Length of Bwd Packets
- Fwd Packet Length Max
- Fwd Packet Length Min
- Fwd Packet Length Mean
- Fwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packet Length Min
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Flow Bytes/s
- Flow Packets/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Total
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Total
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Fwd URG Flags
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Bwd Packets/s
- Min Packet Length
- Max Packet Length
- Packet Length Mean
- Packet Length Std
- Packet Length Variance
- FIN Flag Count
- SYN Flag Count
- RST Flag Count
- PSH Flag Count
- ACK Flag Count
- URG Flag Count
- CWE Fla

In [44]:
### Train/Test Split ###
from sklearn.model_selection import train_test_split

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

X = df.drop(columns=[label_col]).copy()
y = df[label_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    shuffle=True,
)

print(f"Train/Test sizes: {len(y_train)}/{len(y_test)}")

Train/Test sizes: 146292/36573


In [45]:
### Scaling (reuse scaler of source dataset) ###

# no need to scale; benign lab-captured data was scaled during capture, and
# attack samples were scaled earlier.

In [46]:
### Label encoding (reuse encoder of shared label space) ###

ENCODER_PATH = Path("models/label_encoder.joblib")
if not ENCODER_PATH.exists():
    raise FileNotFoundError(f"Label encoder not found at {ENCODER_PATH}")

le = joblib.load(ENCODER_PATH)

# Ensure train/test labels are fully compatible with source-fitted encoder.
raw_train_labels = y_train.astype("string").str.strip()
raw_test_labels = y_test.astype("string").str.strip()
all_unknown = sorted((set(raw_train_labels.dropna()) | set(raw_test_labels.dropna())) - set(le.classes_))
if all_unknown:
    raise ValueError(
        f"Found labels not present in fitted source encoder: {all_unknown[:10]} "
        f"(total={len(all_unknown)})."
    )

y_train = pd.Series(le.transform(raw_train_labels), index=y_train.index, name="Label")
y_test = pd.Series(le.transform(raw_test_labels), index=y_test.index, name="Label")

print(f"Loaded label encoder from {ENCODER_PATH}")
print(f"Encoded classes ({len(le.classes_)}): {list(le.classes_)}")

Loaded label encoder from models/label_encoder.joblib
Encoded classes (13): ['Benign', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Infiltration', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


In [47]:
### Calculate and export covariance and mean statistics ###
# Note: calculated from target training split only.

# Load shared feature space contract.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

# Reuse unified parser so feature-space JSON is handled consistently across cells.
feature_order = load_feature_order(shared_feature_space_path)

# Improvement support: persist canonical feature ordering inside CORAL stats so
# evaluation can verify source/target schema parity before adaptation.
X_target_train_aligned = X_train[feature_order]

# Convert to numpy for CORAL math using float64 for better numerical stability.
X_trg = X_target_train_aligned.to_numpy(dtype=np.float64)

# 1) Feature-wise mean vector.
target_feature_mean = np.mean(X_trg, axis=0)

# 2) Centered target training data.
X_trg_centered = X_trg - target_feature_mean

# Improvement #1: use Ledoit-Wolf shrinkage covariance for a better-conditioned
# covariance estimate than plain sample covariance.
target_cov_estimator = LedoitWolf()
target_cov_estimator.fit(X_trg_centered)
target_covariance = np.asarray(target_cov_estimator.covariance_, dtype=np.float64)
target_covariance = (target_covariance + target_covariance.T) / 2.0

# Save diagnostics consumed by training/eval for spectral-floor and stability analysis.
target_eigenvalues = np.linalg.eigvalsh(target_covariance)
target_min_eig = float(target_eigenvalues.min())
target_max_eig = float(target_eigenvalues.max())
target_cov_condition_number = float(np.linalg.cond(target_covariance))
target_covariance_ridge = 0.0

# Sanity checks.
assert target_covariance.shape[0] == target_covariance.shape[1], "Covariance matrix must be square"
assert target_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

# Package CORAL statistics.
coral_target_stats = {
    "feature_order": feature_order,
    "mean": target_feature_mean,
    "covariance": target_covariance,
    "covariance_estimator": "LedoitWolf",
    "covariance_shrinkage": float(target_cov_estimator.shrinkage_),
    "min_eigenvalue_before_regularization": target_min_eig,
    "max_eigenvalue": target_max_eig,
    "covariance_condition_number": target_cov_condition_number,
    "covariance_ridge": target_covariance_ridge,
}

# Persist for downstream domain adaptation pipeline.
coral_stats_path = Path("models/coral_target_stats.joblib")
coral_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(coral_target_stats, coral_stats_path)

print("CORAL target statistics extracted and saved successfully.")
print("Verified: CORAL stats were computed from target train split only.")
print(f"Train rows used for CORAL: {len(X_target_train_aligned)}")
print(f"Saved to: {coral_stats_path}")
print(f"Features: {len(feature_order)}")
print(f"Covariance shape: {target_covariance.shape}")
print(f"Covariance estimator: LedoitWolf (shrinkage={target_cov_estimator.shrinkage_:.6f})")
print(f"Covariance eigenvalues: min={target_min_eig:.6e}, max={target_max_eig:.6e}")
print(f"Covariance condition number: {target_cov_condition_number:.6e}")

CORAL target statistics extracted and saved successfully.
Verified: CORAL stats were computed from target train split only.
Train rows used for CORAL: 146292
Saved to: models/coral_target_stats.joblib
Features: 70
Covariance shape: (70, 70)
Covariance estimator: LedoitWolf (shrinkage=0.257568)
Covariance eigenvalues: min=1.034795e+05, max=2.091058e+07
Covariance condition number: 2.020746e+02


In [48]:
### Export processed data ###

# Create output directory for processed target train/test splits.
output_dir = Path("data/processed/target")
output_dir.mkdir(parents=True, exist_ok=True)

# Save processed target train data.
train_df = pd.DataFrame(X_train, columns=X_train.columns)
train_df["Label"] = y_train.values
train_df.to_csv(output_dir / "train.csv", index=False)

# Save processed target test data.
test_df = pd.DataFrame(X_test, columns=X_test.columns)
test_df["Label"] = y_test.values
test_df.to_csv(output_dir / "test.csv", index=False)

print("Saved datasets:")
print(f"  Train: {len(train_df)} samples")
print(f"  Test: {len(test_df)} samples")
print(f"  Output directory: {output_dir}")

Saved datasets:
  Train: 146292 samples
  Test: 36573 samples
  Output directory: data/processed/target
